## Import packages

In [1]:
import os
import sys
import json
import argparse
import numpy as np
import math
from einops import rearrange
import time
import random
import string
import h5py
from tqdm import tqdm
import webdataset as wds
import gc

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms

# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True

## Configuration

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
data_type = torch.float16 # change depending on your mixed_precision
num_devices = torch.cuda.device_count()
batch_size = 32
num_epochs=12

print(f"data_type={data_type}, num_devices={num_devices}, batch_size={batch_size}, num_epochs={num_epochs}\n")

data_path = "/teamspace/studios/this_studio/nsd"
subj = 1
subj_list = [subj]
num_sessions = 2
num_test = 1985
num_voxels_list = []

num_samples_per_epoch = (750 * num_sessions) // num_devices 
num_iterations_per_epoch = num_samples_per_epoch // (batch_size * len(subj_list))

def my_split_by_node(urls): 
    return urls

print(f"data_path={data_path}, subj={subj}, subj_list={subj_list}, \n\
num_sessions={num_sessions}, num_test={num_test}, num_voxels_list={num_voxels_list}, \n\
num_iterations_per_epoch={num_iterations_per_epoch}")

data_type=torch.float16, num_devices=1, batch_size=32, num_epochs=12

data_path=/teamspace/studios/this_studio/nsd, subj=1, subj_list=[1], 
num_sessions=2, num_test=1985, num_voxels_list=[], 
num_iterations_per_epoch=46


## Creating wds dataloader

In [3]:
train_data = {}
train_dl = {}
num_voxels = {}
voxels = {}

for s in subj_list:

    train_url = f"{data_path}/wds/subj0{s}/train/" + "{0.." + f"{num_sessions-1}" + "}.tar"
    
    train_data[f'subj0{s}'] = wds.WebDataset(train_url,resampled=True,nodesplitter=my_split_by_node)\
                        .shuffle(750, initial=1500, rng=random.Random(42))\
                        .decode("torch")\
                        .rename(behav="behav.npy", past_behav="past_behav.npy", future_behav="future_behav.npy", olds_behav="olds_behav.npy")\
                        .to_tuple(*["behav", "past_behav", "future_behav", "olds_behav"])

    train_dl[f'subj0{s}'] = torch.utils.data.DataLoader(train_data[f'subj0{s}'], batch_size=batch_size, shuffle=False, drop_last=True, pin_memory=True)

    f = h5py.File(f'{data_path}/betas_all_subj0{s}_fp32_renorm.hdf5', 'r')
    betas = f['betas'][:]
    betas = torch.Tensor(betas).to("cpu").to(data_type)
    num_voxels_list.append(betas[0].shape[-1])
    num_voxels[f'subj0{s}'] = betas[0].shape[-1]
    voxels[f'subj0{s}'] = betas

    print(f"num_voxels for subj0{s}: {num_voxels[f'subj0{s}']}\n")

print("Loaded all subj train dls and betas!\n")

test_url = f"{data_path}/wds/subj0{subj}/test/" + "0.tar"

test_data = wds.WebDataset(test_url,resampled=False,nodesplitter=my_split_by_node)\
                    .shuffle(750, initial=1500, rng=random.Random(42))\
                    .decode("torch")\
                    .rename(behav="behav.npy", past_behav="past_behav.npy", future_behav="future_behav.npy", olds_behav="olds_behav.npy")\
                    .to_tuple(*["behav", "past_behav", "future_behav", "olds_behav"])
test_dl = torch.utils.data.DataLoader(test_data, batch_size=num_test, shuffle=False, drop_last=True, pin_memory=True)

print(f"Loaded test dl for subj{subj}!\n")

num_voxels for subj01: 15724

Loaded all subj train dls and betas!

Loaded test dl for subj1!



In [4]:
f = h5py.File(f'{data_path}/coco_images_224_float16.hdf5', 'r')
# images = f['images'][:] # if you go OOM you can remove the [:] so it isnt preloaded to cpu! (will require a few edits elsewhere though)
# images = torch.Tensor(images).to(device).to(data_type)
images = f['images']
images.shape

(73000, 3, 224, 224)

## Load models

### CLIP image embeddings model

In [5]:
import clip
from PIL import Image

clip_embedder, preprocess = clip.load("ViT-B/32", device=device)

### ImageToBrain

In [6]:
# Input: Clip latents of image
# Output: Brain representation

class ImageToBrain(torch.nn.Module):

    def __init__(self, input_sizes, out_features):
        super(ImageToBrain, self).__init__()
        self.out_features = out_features
        self.linears = torch.nn.Linear(input_sizes, out_features)
    def forward(self, x):
        out = self.linears(x)
        return out

In [7]:
# input_dim = clip_embedder.visual.input_resolution  # Size of the CLIP embedding
input_dim = 512
output_dim = num_voxels[f'subj0{subj}']  # Number of voxels as target output
model = ImageToBrain(input_dim, output_dim).to(device)

## Main

### Preprocess

In [8]:
train_dls = [train_dl[f'subj0{s}'] for s in subj_list]

In [9]:
def pad_batch(batch, target_size, padding_value=0):
    # Calculate how much padding is needed
    padding_needed = target_size - batch.size(0)
    if padding_needed > 0:
        # Create a padding tensor of the same dimension except for the batch size dimension
        padding_tensor = torch.zeros((padding_needed,) + batch.shape[1:], dtype=batch.dtype, device=batch.device)
        # Append the padding tensor to the original batch
        batch = torch.cat([batch, padding_tensor], dim=0)
    return batch

In [10]:
def preprocess():
    voxel_iters = {} # empty dict because diff subjects have differing # of voxels
    image_iters = torch.zeros(num_iterations_per_epoch, batch_size*len(subj_list), 3, 224, 224).float()
    
    for s, train_dl in enumerate(train_dls):
        with torch.cuda.amp.autocast(dtype=data_type):
            for iter, (behav0, past_behav0, future_behav0, old_behav0) in enumerate(train_dl):
                # Get indices and ensure they are sorted and unique
                image_sorted_idx = behav0[:,0,0].cpu().long().numpy()
                image_sorted_idx = np.unique(np.sort(image_sorted_idx))  # Sort and remove duplicates

                # Fetch images using the sorted and unique indices
                image0 = images[image_sorted_idx]
                image0 = torch.tensor(image0, dtype=torch.float16, device=device)  # Convert to tensor

                image0 = pad_batch(image0, 32)

                image_iters[iter, s*batch_size:s*batch_size+batch_size] = image0

                # Similar process for voxel indices
                voxel_sorted_idx = behav0[:,0,5].cpu().long().numpy()
                voxel_sorted_idx = np.unique(np.sort(voxel_sorted_idx))  # Sort and remove duplicates

                # Fetch and store voxel data
                voxel0 = voxels[f'subj0{subj_list[s]}'][voxel_sorted_idx]
                voxel0 = torch.tensor(voxel0, dtype=torch.float16)  # Convert to tensor
                voxel0 = pad_batch(voxel0, 32)

                voxel_iters[f"subj0{subj_list[s]}_iter{iter}"] = voxel0

                if iter >= num_iterations_per_epoch-1:
                    print(f"iter is greater than or equal to num_iterations_per_epoch-1")
                    break
    
    return voxel_iters, image_iters

voxel_iters, image_iters = preprocess()

/tmp/ipykernel_26374/3906065816.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  voxel0 = torch.tensor(voxel0, dtype=torch.float16)  # Convert to tensor


iter is greater than or equal to num_iterations_per_epoch-1


### Train Loop

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()  # Mean Squared Error Loss

def train():

    for epoch in range(num_epochs):
        for train_i in range(num_iterations_per_epoch):
            with torch.cuda.amp.autocast(dtype=data_type):
                optimizer.zero_grad()
                loss=0.

                voxel_list = [voxel_iters[f"subj0{s}_iter{train_i}"].detach().to(device) for s in subj_list]
                voxel_batch = torch.stack(voxel_list)
                voxel_batch = voxel_batch.view(-1, voxel_batch.shape[-1])

                image = image_iters[train_i].detach()
                image = image.to(device)
                clip_latent = clip_embedder.encode_image(image)

                voxel_ridge = model(clip_latent)

                loss = criterion(voxel_ridge, voxel_batch)

                loss.backward()
                optimizer.step()

                print(f"Iteration {train_i+1}/{num_iterations_per_epoch}, Loss: {loss.item()}")

train()


Iteration 1/46, Loss: 1.06444251537323
Iteration 2/46, Loss: 1.053275227546692
Iteration 3/46, Loss: 0.9828944206237793
Iteration 4/46, Loss: 0.9951275587081909
Iteration 5/46, Loss: 0.9871819615364075
Iteration 6/46, Loss: 0.9992367625236511
Iteration 7/46, Loss: 0.9785869121551514
Iteration 8/46, Loss: 1.1316508054733276
Iteration 9/46, Loss: 0.9690160155296326
Iteration 10/46, Loss: 0.9619767665863037
Iteration 11/46, Loss: 0.9917208552360535
Iteration 12/46, Loss: 1.029529094696045
Iteration 13/46, Loss: 1.019917607307434
Iteration 14/46, Loss: 0.9591293334960938
Iteration 15/46, Loss: 0.9804890155792236
Iteration 16/46, Loss: 0.9639966487884521
Iteration 17/46, Loss: 0.9449044466018677
Iteration 18/46, Loss: 0.9227176904678345
Iteration 19/46, Loss: 1.077759861946106
Iteration 20/46, Loss: 0.9874133467674255
Iteration 21/46, Loss: 1.0813952684402466
Iteration 22/46, Loss: 1.0430303812026978
Iteration 23/46, Loss: 1.0145403146743774
Iteration 24/46, Loss: 1.016601324081421
Iteratio

### Eval Loop